# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 7: Fine-tuning LLMs</font>

# <font color="#003660">Notebook 2: Supervised Fine-Tuning LLMs</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know the basics of quantization, parameter-efficient fine-tuning, and supervised fine-tuning. <br>
        ... are able to supervised fine-tune an LLM using the Transformers, PEFT and TRL libraries from huggingface. <br>
        ... know how to teach models to reason (&lt;think&gt;).
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:


* [TRL QLoRA Examples](https://github.com/huggingface/trl/blob/main/examples/notebooks/sft_trl_lora_qlora.ipynb)
* [TRL Examples](https://github.com/huggingface/trl/tree/main/examples)
* All papers referred to in the notebook

## Aligning LLMs

In general, aligning language models to human, organizational, and social values as well as the task (also referred to as LLM alignment) ([Askell et al., 2021](https://doi.org/10.48550/arXiv.2112.00861)) LLMs is usually done using the three steps below.

First, high-qualty human demonstration data is sampled and used for *Supervised Fine-Tuning (SFT)* the foundation model (the pre-trained LLM ([Bommasani et al., 2021](https://doi.org/10.48550/arXiv.2108.07258))). Then comparison data is collected and a reward model is trained. Finally, the reward model is used in proximal policiy optimization to incorporate human feedback, called *Reinforcement Learning from Human Feedback (RLHF)* ([Ouyang et al., 2022](https://proceedings.neurips.cc/paper_files/paper/2022/file/b1efde53be364a73914f58805a001731-Paper-Conference.pdf)) shown in the left of the image below.

![RLHF](https://huyenchip.com/assets/pics/rlhf/6-sft-rlhf.png)

With RLHF, diverse methodologies have been developed. One less resource-intensive and much simpler strategy is *Direct Preference Optimization (DPO)* ([Rafailov et al., 2023](https://doi.org/10.48550/arXiv.2305.18290)) shown in the right of the above image.

## Supervised Fine-Tuning (SFT)

SFT can be done using all kind of high-quality instruction, conversational or prompting data. Its purpose is to train the conversational or instruction-solving manner into the weights of the LLM ([Ouyang et al. 2022](https://proceedings.neurips.cc/paper_files/paper/2022/file/b1efde53be364a73914f58805a001731-Paper-Conference.pdf))

We will now first learn how to supervised fine-tune an LLM while using *Parameter efficient fine-tuning (PEFT)* ([Liu et al., 2022](https://proceedings.neurips.cc/paper_files/paper/2022/file/0cde695b83bd186c1fd456302888454c-Paper-Conference.pdf)) and *Quantization* ([Shkolnik et al., 2020](https://proceedings.neurips.cc/paper/2020/file/3948ead63a9f2944218de038d8934305-Paper.pdf)).

### Installing and loading libraries

Today we will use the libraries:

* [Datasets](https://huggingface.co/docs/datasets/index) for loading and transforming datasets
* [Transformers](https://huggingface.co/docs/transformers/index) transformers to load and train models
* [BitsAndBytes](https://huggingface.co/docs/bitsandbytes/index) to quantize models (more on quantization later)
* [PEFT](https://huggingface.co/docs/peft/index) to parameter-efficient fine-tune the LLMs
* [TRL](https://huggingface.co/docs/trl/index) to direct preference optimize the LLMs
* [Hugging Face Hub](https://huggingface.co/docs/huggingface_hub/index) to share our fine-tuned LLMs

The same procedure as every year - installing and loading packages.

In [ ]:
!pip install -U datasets transformers accelerate bitsandbytes peft trl wandb trackio

In [ ]:
!git config --global credential.helper store # some stuff for huggingface login

You will need a huggingface token to use today session's llama 3.2. You can generate one on your [Account](https://huggingface.co/settings/account)

In [ ]:
import os

import torch

from datasets import load_dataset

from transformers import set_seed, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from huggingface_hub import login, notebook_login

from peft import LoraConfig

from trl import SFTConfig, SFTTrainer

notebook_login()

In [ ]:
# helper function
def generate(messages, model):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0]
    return response

### Teaching Llama to `<think>`

We already worked with reasoning models such as [Qwen3](https://doi.org/10.48550/arXiv.2505.09388), which is a reasoning LLM from Alibaba ([Yang et al., 2025](https://doi.org/10.48550/arXiv.2505.09388)). These are special LLMs trained to first reason step-by-step similar to Chain-of-Thought you learned last session. This training approach shifts LLMs away from fast, error-prone (System 1) responses toward more deliberate, reflective (System 2) reasoning about the task ([Li et a., 2025](https://doi.org/10.48550/arXiv.2502.17419)).

This is especially helpful when we want to do synthesis of information.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-0.6B",
    dtype="auto",
    device_map="cuda",
)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    dtype="auto",
    device_map="cuda",
)
print(generate([{"role": "user", "content": "What is the meaning of life?"}], model))

As you can see this model generates so-called thinking tags: `<think>...whatever to think of here...</think>`.

Let's clear GPU memory by restarting the kernel:

In [ ]:
os.kill(os.getpid(), 9)

Let's try this with [llama-3.2](https://ai.meta.com/blog/llama-3-2-connect-2024-vision-edge-mobile-devices/).

In [ ]:
import os

import torch

from datasets import load_dataset

from transformers import set_seed, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from peft import LoraConfig

from trl import SFTConfig, SFTTrainer

# helper function
def generate(messages, model):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0]
    return response

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    token=True,
)
print(generate([{"role": "user", "content": "What is the meaning of life?"}], base_model))

Hmm... It is not thinking and no `<think>` tags. We could prompt it to do so using Chain-of-Thought prompting but it will not be the same.

So we fine-tune it to really `<think>`.

But if you check out the GPU usage in the right, then you will see that the model needs 12.7 GB of GPU-RAM to run. That's much.

**The problem:** large language models are usually **large**.

Let's check out how many GB GPU-RAM the model will need for classical fine-tuning:

In [ ]:
!accelerate estimate-memory --library_name transformers meta-llama/Llama-3.2-3B-Instruct

48GB! Even 2 Sodalab computers would not be able to handle this..

Nevertheless we want to fine-tune the LLM (yeah I know, its actually an SLM).

Thus, we need to introduce some amazing concepts.

But first let's restart the notebook one last time:

In [ ]:
os.kill(os.getpid(), 9)

And again load all the packages required:

In [ ]:
import os

import torch

from datasets import load_dataset

from transformers import set_seed, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from peft import LoraConfig

from trl import SFTConfig, SFTTrainer

# helper function
def generate(messages, model):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0]
    return response

Now we can talk about amazing concepts to make fine-tuning available everywhere.

First let's talk about quantization.

### Quantization

*Quantization* refers to reducing the precision of floating point numbers in mathematical operations ([Nagel et a., 2021](https://doi.org/10.48550/arXiv.2106.08295)).

![FP8](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/blog/bitsandbytes/FP8-scheme.png)

(Image sources: [sgugger](https://huggingface.co/sgugger), [BitsAndBytes FP4 Blog Post](https://huggingface.co/blog/4bit-transformers-bitsandbytes))

We can do this by the code below.

In [ ]:
set_seed(42)

In [ ]:
# loading the quantization config, 4-bit mode currently one of the smallest
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Load the model in 4-bit mode
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config, # here we apply quantization
    device_map="auto",
    trust_remote_code=True,
    token=True,
)

And run our quantized model to check if the quantization worked well.

In [ ]:
print("INPUT")
chat_prompt = tokenizer.apply_chat_template([{"role": "user", "content": "how can i develop a habit of drawing daily"}], tokenize=False, add_generation_prompt=True)
print(chat_prompt)
inputs = tokenizer(chat_prompt, return_tensors="pt").to("cuda")
res = base_model.generate(**inputs, max_new_tokens=128, pad_token_id=tokenizer.pad_token_id)

print("-----")

print("OUTPUT:")
print(tokenizer.decode(res[0]).replace(chat_prompt, ""))

Wow, it worked. And if you checkou the GPU usage, you will see only 3.6 GB GPU-RAM usage.

But no thinking at all in the model...

Let's checkout this amazing dataset that enables thinking in diverse languages.

In [ ]:
dataset_name = "HuggingFaceH4/Multilingual-Thinking"
train_dataset = load_dataset(dataset_name, split="train")

In [ ]:
import json
print(json.dumps(train_dataset[1], indent=4))

As you can see there are the typical messages (we will talk about this later) and there is also a key for `analysis`. This is the reasoning (or `<think>...</think>`).

Let's process the dataset so that the message content contains the analysis as thinking.

In [ ]:
def merge_thinking_and_remove_key(example):
    new_messages = []
    for msg in example["messages"]:
        content = msg["content"]
        thinking = msg.pop("thinking", None)
        if thinking and isinstance(thinking, str) and thinking.strip():
            content = f"\n<think>\n{thinking}\n</think>\n\n{content}" # add thinking here
        msg["content"] = content
        new_messages.append(msg)
    example["messages"] = new_messages
    return example

train_dataset = train_dataset.map(merge_thinking_and_remove_key)

When we now apply the chat template, then we see that the `<think>`-tags are in there.

In [ ]:
print(tokenizer.apply_chat_template(train_dataset[0]["messages"], tokenize=False))

Nice, so let's fine-tune the LLM. But can we simply fine-tune quantized model?

Short answer - no we can't. But we can use the principle of adapters to apply low-rank adaptation (LoRA).

### (Q-)LoRA

*(Quantized) Low Rank Adaptation ((Q)LoRA)* is a concept first introduced by [Hu et al. (2021)](https://openreview.net/forum?id=nZeVKeeFYf9), shown in the image from their paper below.

![LoRA](https://media.datacamp.com/legacy/v1705430151/image4_b814637cd2.png)

This was extended with quantization by [Dettmers et al. (2023)](https://dl.acm.org/doi/10.5555/3666122.3666563), shown in the image below (also ).

![(Q)Lora](https://miro.medium.com/v2/resize:fit:720/format:webp/1*tMaufQKw0Boq4pYlOWDLBg.png)

(Here is a link to the [Virtual Poster](https://neurips.cc/virtual/2023/poster/71815) and this helpful article for [LoRA](https://ritvik19.medium.com/papers-explained-lora-a48359cecbfa))

To add LoRA adapters we need to set the ``lora_alpha`` and the ``lora_r`` (lora_r=rank, scaling factor of the wight matrices $\frac{lora_{alpha}}{lora_{rank}}$) and the ``lora_dropout``(determines the dropout of the linear adapter layers).

Finalls we need to determine, which linear layers of the LLM should be extended by adapters (usualy the projection layers (linear layers of the attention heads) $W_q$ (query) and $W_v$ (value), sometimes $W_k$ (key)).

Here we apply QLoRA (quantized, because we use a quantizen model).

In [ ]:
peft_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
)

Here we are defining many parameters for the supervised fine-tuning. You can check them out [here](https://huggingface.co/docs/trl/sft_trainer#trl.SFTConfig).

In [ ]:
output_dir = MODEL_NAME.split("/")[-1] + "-Thinking"
training_args = SFTConfig(
    # Training schedule / optimization
    per_device_train_batch_size = 1,      # Batch size per GPU
    gradient_accumulation_steps = 1,      # Gradients are accumulated over multiple steps → effective batch size = 2 * 8 = 16
    warmup_steps = 5,
    num_train_epochs = 1,               # Number of full dataset passes. For shorter training, use `max_steps` instead (this case)
    #max_steps = 100,
    learning_rate = 2e-4,                 # Learning rate for the optimizer
    optim = "paged_adamw_8bit",           # Optimizer

    # Logging / reporting
    logging_steps=25,                      # Log training metrics every N steps
    report_to="none",                  # Experiment tracking tool
    trackio_space_id=output_dir,          # HF Space where the experiment tracking will be saved
    output_dir=output_dir,                # Where to save model checkpoints and logs

    max_length=4096,                      # Maximum input sequence length
    use_liger_kernel=False,                # Enable Liger kernel optimizations for faster training
    activation_offloading=False,           # Offload activations to CPU to reduce GPU memory usage
    gradient_checkpointing=True,          # Save memory by re-computing activations during backpropagation

    # Hub integration
    push_to_hub=True,                     # Automatically push the trained model to the Hugging Face Hub
                                          # The model will be saved under your Hub account in the repository named `output_dir`

    gradient_checkpointing_kwargs={"use_reentrant": False}, # To prevent warning message
)

In [ ]:
trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config
)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Now after the definition of the dataset, SFT training arguments, PEFT and LoRA we can start the training.

In [ ]:
trainer_stats = trainer.train()

As you can see, the GPU-RAM utilization is only at 14.5 GB. With this, you can run training for 4096 tokens (approx. 3072 or around 8 pages of text in word).

Let's check out the model:

In [ ]:
from transformers import pipeline

In [ ]:
generator = pipeline("text-generation", model=f"skaltenp/{output_dir}", device="cuda")

In [ ]:
set_seed(42)
question = "What is a large language model?"
output = generator([{"role": "user", "content": question}], max_new_tokens=128, return_full_text=True)[0]
print(output["generated_text"][-1]["content"])

## Merge and Unload the Model

When we have applied (Q)LoRA, we maybe want to train the model with other data, or (as in our case) proceed with the next step of DPO. Then we need to merge the model weights first, to get one model without adapters.

![Lora Merging](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/peft/lora_diagram.png)

In [ ]:
from peft import AutoPeftModelForCausalLM

#del base_model

torch.cuda.empty_cache()

model = AutoPeftModelForCausalLM.from_pretrained(
    f"skaltenp/{output_dir}",
    device_map="auto",
    dtype=torch.bfloat16
    # do NOT load the model in 4-bit precision
    # this will lead to rounding errors
)
model = model.merge_and_unload()

model.push_to_hub(
    f"skaltenp/{output_dir}-merged",
    safe_serialization=True
)

# do not forget the tokenizer
tokenizer.push_to_hub(f"skaltenp/{output_dir}-merged")

### Key findings:
* SFT needs high quality human data
* To run models as well as SFT on small GPUs you can use Quanitization and (Q)LoRA
* To have one model instead of adapters on an existing model, you can merge weights

But:
* Out there are diverse other approaches of PEFT (e.g., [REFT](https://github.com/stanfordnlp/pyreft)).
* Out there are other interesting fine-tuning strategies (e.g., [Prefix-Tuning](https://doi.org/10.48550/arXiv.2101.00190))
* (Q)LoRA only changes the style the LLM answers. It is usually not the best application for integrating new knowledge ([Jiang et al., 2024](https://doi.org/10.48550/arXiv.2405.12130); [Shuttleworth et al., 2024](https://doi.org/10.48550/arXiv.2410.21228)).